# **Modelo clasificatorio de calidad de champiñones**

In [1]:
# Instalación de librerías básicas
!pip install torch torchvision pandas numpy matplotlib scikit-learn seaborn

In [2]:
import pandas as pd

def load_data_from_github():
    try:
        #URLs Raw de archivos de github
        dataset_url = "https://raw.githubusercontent.com/VivianaPM/DL-VisNIR-QualityClassifier/refs/heads/feature/model-mushroom/Mushroom/Data/Dataset_Mus.csv"
        rangos_url = "https://raw.githubusercontent.com/VivianaPM/DL-VisNIR-QualityClassifier/refs/heads/feature/model-mushroom/Mushroom/Data/MushQualTags.csv"

        print("Cargando datos...")
        
        dataset = pd.read_csv(dataset_url, sep=',')
        rangos = pd.read_csv(rangos_url, sep=';')
        
        print(f"Dataset cargado: {dataset.shape}")
        print(f"Rangos cargados: {rangos.shape}")
        
        return dataset, rangos
        
    except Exception as e:
        print(f"Error: {e}")
        return None, None

In [3]:
#@title CARGAR DATOS DESDES GITHUB
df_Mush, rank_Mush = load_data_from_github()

if 'Dry matter' not in df_Mush.columns:
    print("Busca la columna equivalente a 'Dry matter' en:")
    print(df_Mush.columns.to_list())

Cargando datos...
Dataset cargado: (250, 206)
Rangos cargados: (6, 3)


In [4]:
display(rank_Mush)

,Category,Min_DryM,Max_DryM
0,Excelente,0.335,0.970
1,Muy buena,0.180,0.335
2,Buena,0.125,0.180
3,Razonable,0.089,0.125
4,Mala calidad,0.080,0.089
5,Muy mala,0.000,0.080


In [8]:
#@title ASIGNAR CATEGORÍA REAL
def define_categories_mush(dry_matter, rank_Mush):
    for _, fila in rank_Mush.iterrows():
        if fila['Min_DryM'] <= dry_matter <= fila['Max_DryM']:
            return fila['Category']
    return 'Desconocida'

df_Mush['Category'] = df_Mush['Dry matter'].apply(
    lambda x: define_categories_mush(x, rank_Mush)
)

In [10]:
#@title DISTRIBUCIÓN DE CATEGORIAS
print(df_Mush['Category'].value_counts())

Category
Excelente       63
Razonable       62
Muy mala        51
Muy buena       36
Buena           26
Mala calidad    12
Name: count, dtype: int64


In [14]:
#@title CODIFICACIÓN TEMPORAL PARA ESTRATIFICAR
category_map_mush = {
    'Excelente': 0,
    'Razonable': 1,
    'Muy mala': 2,
    'Muy buena': 3,
    'Buena': 4,
    'Mala calidad': 5

}

df_Mush['strat_label'] = df_Mush['Category'].map(category_map_mush)

# Definición de X y Y globales
X = df_Mush.drop(columns=['Category', 'strat_label'])
y = df_Mush['strat_label']

# CHEKPOINT: REVISAR CODIFICACIÓN TEMPORAL
print(df_Mush[['Category', 'strat_label']].head())

       Category  strat_label
0  Mala calidad            5
1      Muy mala            2
2      Muy mala            2
3      Muy mala            2
4      Muy mala            2
